# 05 — Compute Stats

Two-pass approach:
1. Compute raw axis scores for all 4 personas simultaneously
2. Calibrate via min-max normalisation to Section 10 target ranges
3. Apply simulated this_week deltas

Output: `output/stats_{persona}.json` with baseline + this_week 5-axis scores.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))
from stats import compute_raw_scores, calibrate_scores, compute_week_scores

OUTPUT = Path('..') / 'output'
PERSONAS = ['aisyah', 'daniel', 'mei_ling', 'hafiz']

In [ ]:
# Simulated this_week transaction overrides (Section 10 demo scenarios)
SIMULATED_WEEK = {
    'aisyah': [
        {'timestamp': '2025-03-27T08:30:00', 'category': 'food_groceries',  'amount': 45.0,  'is_recurring': False},
        {'timestamp': '2025-03-28T12:00:00', 'category': 'savings_transfer','amount': 70.0,  'is_recurring': True},
        {'timestamp': '2025-03-29T19:00:00', 'category': 'food_mamak',      'amount': 12.0,  'is_recurring': False},
        {'timestamp': '2025-03-30T10:00:00', 'category': 'transport_tng',   'amount': 30.0,  'is_recurring': False},
    ],
    'daniel': [
        {'timestamp': '2025-03-27T12:00:00', 'category': 'food_grab',       'amount': 32.0,  'is_recurring': False},
        {'timestamp': '2025-03-27T20:00:00', 'category': 'food_grab',       'amount': 28.0,  'is_recurring': False},
        {'timestamp': '2025-03-28T12:00:00', 'category': 'food_grab',       'amount': 35.0,  'is_recurring': False},
        {'timestamp': '2025-03-29T10:00:00', 'category': 'shopping_online_shopee', 'amount': 180.0, 'is_recurring': False},
        {'timestamp': '2025-03-29T20:00:00', 'category': 'food_grab',       'amount': 30.0,  'is_recurring': False},
        {'timestamp': '2025-03-30T12:00:00', 'category': 'food_grab',       'amount': 27.0,  'is_recurring': False},
        {'timestamp': '2025-03-31T08:00:00', 'category': 'subscription',    'amount': 15.98, 'is_recurring': True},
    ],
    'mei_ling': [
        {'timestamp': '2025-03-27T09:00:00', 'category': 'food_groceries',  'amount': 38.0,  'is_recurring': False},
        {'timestamp': '2025-03-28T10:00:00', 'category': 'utilities',       'amount': 55.0,  'is_recurring': True},
        {'timestamp': '2025-03-29T14:00:00', 'category': 'savings_transfer','amount': 80.0,  'is_recurring': False},
        {'timestamp': '2025-03-30T11:00:00', 'category': 'food_mamak',      'amount': 10.0,  'is_recurring': False},
    ],
    'hafiz': [
        {'timestamp': '2025-03-27T08:00:00', 'category': 'food_groceries',  'amount': 95.0,  'is_recurring': False},
        {'timestamp': '2025-03-28T10:00:00', 'category': 'transport_tng',   'amount': 50.0,  'is_recurring': False},
        {'timestamp': '2025-03-30T12:00:00', 'category': 'food_mamak',      'amount': 75.0,  'is_recurring': False},
        {'timestamp': '2025-03-31T09:00:00', 'category': 'savings_transfer','amount': 1500.0,'is_recurring': True},
    ],
}

In [ ]:
# Pass 1: compute raw scores for all personas
all_transactions = {}
all_raw = {}

for persona in PERSONAS:
    with open(OUTPUT / f'{persona}.json', encoding='utf-8') as f:
        transactions = json.load(f)
    with open(OUTPUT / f'motifs_{persona}.json', encoding='utf-8') as f:
        motif_data = json.load(f)
    all_transactions[persona] = transactions
    all_raw[persona] = compute_raw_scores(transactions, motif_data['motif_strength_score'])

print('Raw scores (pre-calibration):')
axes = ['restraint', 'consistency', 'resilience', 'foresight', 'recovery']
print(f'{"Persona":<12}' + ''.join(f'  {a[:5]:>8}' for a in axes))
print('-' * 60)
for p in PERSONAS:
    print(f'{p:<12}' + ''.join(f'  {all_raw[p][a]:>8.4f}' for a in axes))

In [ ]:
# Pass 2: calibrate to target ranges
calibrated = calibrate_scores(all_raw)

print('Calibrated baseline scores:')
print(f'{"Persona":<12}' + ''.join(f'  {a[:4].upper():>8}' for a in axes))
print('-' * 60)
for p in PERSONAS:
    print(f'{p:<12}' + ''.join(f'  {calibrated[p][a]:>8.1f}' for a in axes))

In [ ]:
# Pass 3: compute this_week scores + write output
all_stats = {}

for persona in PERSONAS:
    with open(OUTPUT / f'motifs_{persona}.json', encoding='utf-8') as f:
        motif_data = json.load(f)

    baseline = calibrated[persona]
    week_scores = compute_week_scores(
        baseline,
        all_transactions[persona],
        motif_data['motif_strength_score'],
        SIMULATED_WEEK[persona],
    )
    stats = {**baseline, **week_scores}
    all_stats[persona] = stats

    out_path = OUTPUT / f'stats_{persona}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)

print('Stats written.')

In [ ]:
# Verification table: baseline (b) vs this_week (w)
print(f'{"Persona":<12}' + ''.join(f'  {a[:4].upper()}(b)  {a[:4].upper()}(w)' for a in axes))
print('-' * 85)
for persona in PERSONAS:
    s = all_stats[persona]
    row = f'{persona:<12}'
    for a in axes:
        row += f'  {s[a]:>6.1f}    {s[a+"_week"]:>6.1f}'
    print(row)

print()
print('Target baseline (Section 10):')
targets = {
    'aisyah':   [78, 82, 70, 75, 65],
    'daniel':   [42, 38, 55, 60, 50],
    'mei_ling': [50, 35, 45, 40, 75],
    'hafiz':    [88, 92, 85, 90, 70],
}
print(f'{"Persona":<12}  {"REST":>6}  {"CONS":>6}  {"RESI":>6}  {"FORE":>6}  {"RECO":>6}')
for persona, vals in targets.items():
    print(f'{persona:<12}  ' + '  '.join(f'{v:>6}' for v in vals))